# VLM-DENTAL - Trace Generation Workspace

This notebook handles autonomous CoT trace generation using LangGraph and vLLM.


## 1. Environment Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**(Optional) Fresh Start Cleanup:**
Run this cell ONLY if you need to completely delete the VLM-DENTAL folder from your Google Drive to start over.

In [ ]:
# Uncomment the line below to delete the folder, then run the cell
# !rm -rf /content/drive/MyDrive/VLM-DENTAL

In [ ]:
import os

# Set this to True to save the 10GB dataset and repo itself to Google Drive.
# Set this to False to keep the repo/dataset in temporary Colab storage.
SAVE_DATASET_AND_CODE_TO_DRIVE = False

# Set this to True to save generated YOLO model outputs to Google Drive.
# Set this to False to keep YOLO weights/results in the active repo clone under /content/.../data/models.
SAVE_YOLO_RESULTS_TO_DRIVE = True

drive_path = "/content/drive/MyDrive/VLM-DENTAL"
colab_path = "/content/VLM-DENTAL"
work_dir = drive_path if SAVE_DATASET_AND_CODE_TO_DRIVE else colab_path
models_root = f"{drive_path}/data/models" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/models"
os.environ["YOLO_MODELS_ROOT"] = models_root

In [ ]:
import os

if SAVE_DATASET_AND_CODE_TO_DRIVE:
    os.chdir("/content/drive/MyDrive")
else:
    os.chdir("/content")

if not os.path.exists("VLM-DENTAL"):
    os.system("git clone https://github.com/rezaxr14/VLM-DENTAL.git")

os.chdir(work_dir)
os.system("git pull")

# Ensure the selected output root exists without pulling everything into Drive by default.
os.makedirs(models_root, exist_ok=True)
if SAVE_YOLO_RESULTS_TO_DRIVE:
    os.makedirs(f"{drive_path}/data/traces", exist_ok=True)

In [ ]:
# Install the project and all its requirements
!pip install -e .
!pip install python-dotenv pandas pillow google-generativeai anthropic huggingface_hub ultralytics

## 2. Configure Credentials (Colab Secrets Tab Support)
Loads API keys and GitHub tokens automatically from Colab Secrets tab (`google.colab.userdata`).

In [ ]:
import os
import shutil

# Ensure .env exists from example
if not os.path.exists('.env') and os.path.exists('.env.example'):
    shutil.copy('.env.example', '.env')

try:
    from google.colab import userdata
    
    # Load Unified Provider Keys
    for key in ['GEMINI_API_KEY', 'NVIDIA_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY']:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass

    # Hugging Face Token
    try:
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            os.environ['HF_TOKEN'] = hf_token
    except Exception:
        pass

    print("Secrets tab checked and loaded.")
except ImportError:
    print("Not running in Colab environment or userdata API unavailable.")

# Ensure critical config for local LangGraph trace generation is set
os.environ['GENERATOR_PROVIDER'] = 'local'
os.environ['GENERATOR_MODEL'] = 'Qwen/Qwen3-VL-8B-Thinking'
os.environ['LOCAL_VLLM_BASE_URL'] = 'http://localhost:8000/v1'


## 3. Dataset Download & Cleanup
Run this to download the dataset if you haven't already. It will extract and structure it automatically.

In [ ]:
!python download_and_cleanup.py

In [ ]:
# Delete unused partial datasets to save space
!rm -rf data/dentex/DENTEX/training_data/disease
!rm -rf data/dentex/DENTEX/training_data/quadrant
!rm -rf data/dentex/DENTEX/training_data/unlabelled/

!rm -rf data/dentex/DENTEX/testing_data/disease
!rm -rf data/dentex/DENTEX/testing_data/quadrant

# Delete corrupted cache folder from any previous bugs (if it exists)
!rm -rf "C:\\Users\\rezax\\dental_agent_cache"

## 4. Install & Stand up vLLM Server
Installs vLLM and starts the OpenAI-compatible API server in the background for Qwen3-VL-8B-Thinking.


In [ ]:
!pip install vllm langgraph
import subprocess
import time
import os
print('Starting vLLM server...')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
vllm_process = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'Qwen/Qwen3-VL-8B-Thinking',
    '--port', '8000',
    '--max-model-len', '4096'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print('vLLM server started in background. Waiting for it to become ready...')
time.sleep(30) # Wait for model to load


## 5. Test LangGraph Loop against vLLM
Tests the LangGraph execution loop to verify tool calling works with the local vLLM server.


In [ ]:
# Run the LangGraph test script on a sample image
import glob
sample_images = glob.glob('data/dentex/DENTEX/training_data/quadrant-enumeration-disease/xrays/*.png')
test_img = sample_images[0] if sample_images else 'data/some_test_image.jpg'
!python scripts/test_langgraph_loop.py --image {test_img} --model Qwen/Qwen3-VL-8B-Thinking


## 6. Autonomous CoT Trace Generation
Runs the daily trace generator and continuously saves progress to `train_cot_traces.jsonl`.


In [ ]:
trace_output = f"{drive_path}/data/traces/train_cot_traces.jsonl" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/traces/train_cot_traces.jsonl"
os.makedirs(os.path.dirname(trace_output), exist_ok=True)
!python scripts/run_trace_gen.py --split train --output {trace_output}


## 7. Download Generated Traces Locally


In [ ]:
from google.colab import files
import os
trace_path = f"{drive_path}/data/traces/train_cot_traces.jsonl" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/traces/train_cot_traces.jsonl"
if os.path.exists(trace_path):
    files.download(trace_path)
else:
    print('Trace file not found.')
